## Notebook for testing gridded stat and anomaly methods

### Import libraries and initiate utilities routines

In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path
import xarray as xr
import os
from datetime import datetime
import matplotlib.pyplot as plt
import geopandas as gpd

current_path = Path.cwd()

# Search for the utilities directory
for parent in current_path.parents:
    candidate = parent / "utilities"
    if candidate.is_dir():
        sys.path.insert(0, str(parent))
        break
else:
    raise FileNotFoundError("Could not locate 'utilities' directory in parent paths.")

from utilities import init_notebook_environment
#from utilities import statanom_utilities, date_utilities, file_utilities, calc_primprod, calc_phytosizeclass, mapping_utilities,
from utilities import get_dataset_products,get_prod_files,dataset_defaults,run_psc_pipeline,subset_dataset, parse_dataset_info,get_dates,get_source_file_dates
from utilities import period_info,resolve_dataset_map,get_dataset_dirs,get_nc_prod
from utilities import get_pyfile_functions,find_function, find_text,build_product_attributes
from utilities import run_stats_pipeline,get_groupby_hint,build_stats_map, parse_dataset_info, file_parser
from utilities import date_utilities, get_dates,get_source_file_dates,get_file_dates,get_default_dataset_product
from utilities import period_utilities, period_info, get_period_info, get_period_sets, get_period_dates, extract_period_code, check_period, generate_periods, date_to_daily_period


env = init_notebook_environment(verbose=False)


In [ ]:
build_stats_map('CHL','M',subset='NES',verbose=True)


In [ ]:
get_prod_files('SST',period='M',map_region='NES')

In [ ]:
files = get_prod_files('CHL',period='M',map_region='NES')
get_period_sets('MONTH', files=files)

In [ ]:
run_stats_pipeline("SST",periods="M",subset="NES",verbose=True,debug=True)

In [ ]:
files = get_prod_files('CHL')
fp = file_parser(files[0])
fp

In [ ]:
files = get_prod_files('SST')
ds = xr.open_mfdataset(files)
ds

In [ ]:
files = get_prod_files('SST',period='M',map_region='NES')
ds = xr.open_mfdataset(files)
da = ds.SST_M_mean
ds

In [ ]:
f = get_prod_files('SST',dataset='CORALSST')
ds = xr.open_mfdataset(f)
ds

In [ ]:
get_nc_prod('CORALSST','SST')

In [ ]:
# --- Quick Test Script ---
prod = "CHL"
per  = "M"  # Monthly Mean
subset = "NES"
dataset = 'OCCCI'

run_stats_pipeline(prod,periods=per,subset=subset,dataset=dataset,verbose=True,debug=False)

In [2]:
run_stats_pipeline('SST',periods=['MONTH'],subset='NES',verbose=True,debug=False)

🤫 Quiet Mode: RuntimeWarnings are hidden.

🚀 Starting pipeline for SST | Period: MONTH
🔍 Resolving output directory for SST (MONTH)...
📦 Auto-resolved dataset_map → MAPPED_2KM_DAILY
📂 Transformed path to CLIMS: /Users/kimberly.hyde/Documents/nadata/DATASETS/ACSPO/V2.81/PRODUCTS/NES_2KM_CLIMS/SST
🔍 DEBUG: Output directory for SST (MONTH) is /Users/kimberly.hyde/Documents/nadata/DATASETS/ACSPO/V2.81/PRODUCTS/NES_2KM_CLIMS/SST
Searching for M files for None:SST
📦 Auto-resolved dataset_map → MAPPED_2KM_DAILY
📂 Transformed path to STATS: /Users/kimberly.hyde/Documents/nadata/DATASETS/ACSPO/V2.81/PRODUCTS/NES_2KM_STATS/SST
🔍 Searching for .nc files in: /Users/kimberly.hyde/Documents/nadata/DATASETS/ACSPO/V2.81/PRODUCTS/NES_2KM_STATS/SST
🧾 Provenance log:
  • Loaded dataset defaults → version: V2.8.1, map: GLOBAL_2KM, default_product: SST
  • Using default dataset_map: GLOBAL_2KM
  • Resolved product path → /Users/kimberly.hyde/Documents/nadata/DATASETS/ACSPO/V2.81/PRODUCTS/NES_2KM_STATS/SST


In [ ]:
files = get_prod_files('SST')
fp = file_parser(files[0])
fp[0]['dataset']

In [ ]:
sm = build_stats_map('SST','M')
sm[0][0]


In [ ]:
# --- Quick Test Script ---
prod = "SST"
per  = "M"  # Monthly Mean
subset = "NES"

print(f"🧪 Testing Pipeline Discovery for {prod}...")

try:
    # 1. Test Directory Resolution
    out_dir = get_prod_files(prod, period=per, map=subset, getfilepath=True)
    print(f"✅ Resolved Output Dir: {out_dir}")

    # 2. Test Map Building
    # Set overwrite=True to force 'is_up_to_date' to False for testing
    s_map = build_stats_map(prod, per, subset=subset, overwrite=True, verbose=True)

    if s_map:
        first_key = list(s_map.keys())[0]
        sample = s_map[first_key]
        print(f"\n📝 Sample Task Details for {first_key}:")
        print(f"   • Output Path: {sample['output']}")
        print(f"   • Input Count: {len(sample['inputs'])} files")
        print(f"   • Needs Update: {not sample['is_up_to_date']}")
    else:
        print("❌ Test Failed: No tasks generated.")

except Exception as e:
    print(f"💥 Test Crashed: {e}")

In [ ]:
stats_map = build_stats_map("SST", "M")
#stats_map
#pm = stats_map['period_map']
#output_periods = list(stats_map["output"].keys())
#output_periods
all_outputs = [info['output'] for info in stats_map.values()]
all_outputs[:3]

#for period, info in pm.items():
#    print(period, info["input_files"])


In [ ]:
get_prod_files('SST')

In [ ]:
path = get_prod_files('SST', period='M', getfilepath=True)


In [ ]:
run_stats_pipeline("SST")

## Demo for PERIOD functions

In [ ]:
# Determine if a period code is a climatology period
from utilities import get_period_info
info = get_period_info()
periods = info.keys()
for p in periods:
    i = info.get(p)
    print(f"{p} is climatology = {i['is_climatology']}")

In [ ]:
# Find files and dates, then get period sets for Months
from utilities import get_dates,get_period_sets, get_period_dates

dates = get_dates([2020,2025])
get_period_sets('W',dates=dates)


In [ ]:
get_period_dates('M_202001')

In [ ]:

perdates = get_period_dates(sets)
perdates
#for a, pd in zip(sets, perdates):
#    print("Week:",a,pd)

In [ ]:
# Find files and dates, then get period sets for SEA
from utilities import get_period_sets
files = get_prod_files("CHL")
dates = get_file_dates(files)
start_dates = [start for start, _ in dates]

sets = get_period_sets('SEA',dates=start_dates)

for a in sets:
    print("SEA periods:",a)

In [ ]:
# Get the dates from the input periods
from utilities import get_period_dates

files = ["D_20250926",
         "DD_20250926_20251231",
         "DOY_023_2020_2025",
         "DD3_20250310_20250312",
         "D8_20250101_20250108",
         "W_202301",
         "WW_202501_202552",
         "WEEK_22_2020_2025",
         "M_202204",
         "MM_202501_202612",
         "MM3_202501_202512",
         "MONTH_11_2020_2025",
         "A_2025",
         "ANNUAL_2020_2025",
         "Y_2025",
         "YY_2020_2025",
         "YEAR_2020_2025"
         ]

#g = get_period_dates(files)
for f in files:
    print(f"{f}: {get_file_dates([f])}")

dates = get_file_dates(files)

date_format="%Y%m%d"
valid = [
        (datetime.strptime(s, date_format), datetime.strptime(e, date_format))
        for s, e in dates
        if s != "NA" and e != "NA"
    ]

min_date = min(s for s, _ in valid)
max_date = max(e for _, e in valid)
print (min_date.strftime(date_format), max_date.strftime(date_format))




In [ ]:
run_stats_pipeline(prods='SST',dataset='CORALSST',periods=['M','MONTH'])

### Daily inputs
Create stats from the daily input files
* Monthly
* Weekly
* 3-day running
* 8-day running

In [ ]:
# Get Chlorophyll Files
get_dataset_products('OCCCI')
print(dataset_defaults()['OCCCI'])
dataset_products = get_dataset_products('OCCCI')
print(dataset_products.keys())
files = get_prod_files('CHL',dataset='OCCCI')
ds = xr.open_mfdataset(files)
ds.data_vars.keys()